In [1]:
import gradio as gr
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from rake_nltk import Rake

/home/apoorva/Desktop/example/code_translation/tools/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
@tool
def extract_keywords(sentence: str) -> str:
    """Extract keywords from a given sentence using RAKE algorithm."""
    rake = Rake()
    rake.extract_keywords_from_text(sentence)
    keywords = rake.get_ranked_phrases()
    return str(keywords) if keywords else "No Keywords"

In [3]:
llm = ChatOllama(model="devstral:24b", base_url="http://localhost:11434")
tools = [extract_keywords]
tools_map = {"extract_keywords": extract_keywords}
llm_with_tools = llm.bind_tools(tools)

In [4]:
def chat(message, history):
    # Build message history
    messages = [SystemMessage(content="You are a very helpful assistant.")]
    for h in history:
        role, content = h["role"], h["content"]
        if role == "user":
            messages.append(HumanMessage(content=content))
        elif role == "assistant":
            messages.append(AIMessage(content=content))
    messages.append(HumanMessage(content=message))

    # First LLM call
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    # Handle tool calls if any
    if response.tool_calls:
        for tool_call in response.tool_calls:
            selected_tool = tools_map[tool_call["name"]]
            result = selected_tool.invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

        # Second LLM call with tool results
        response = llm_with_tools.invoke(messages)

    return response.content

In [5]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
